# Semantic Segmentation Training with GeoAI

GeoAI's `train_segmentation()` trains a segmentation model using [segmentation_models.pytorch](https://github.com/qubvel-org/segmentation_models.pytorch) architectures (U-Net, FPN, DeepLabV3+, etc.) with 500+ pretrained encoder backbones.

The entire workflow — dataset creation, training loop, logging, checkpoint saving — is handled by a single function call backed by PyTorch Lightning.

## References
- GeoAI training: https://geoai.gishub.org/geoai/
- segmentation_models.pytorch: https://github.com/qubvel-org/segmentation_models.pytorch

## 1. Setup — Prepare Chip Dataset

Run `0-overview_and_data.ipynb` first to generate the chips/ directory.

In [ ]:
import os
import geoai
import matplotlib.pyplot as plt
import numpy as np

chip_dir = "chips"
assert os.path.exists(chip_dir), "Run 0-overview_and_data.ipynb first to generate chips."

image_chips = sorted([f for f in os.listdir(os.path.join(chip_dir, "images")) if f.endswith(".tif")])
print(f"Found {len(image_chips)} training chips")

## 2. Create a GeoAI Dataset

In [ ]:
from geoai.train import SegmentationDataset
from torch.utils.data import DataLoader, random_split

dataset = SegmentationDataset(
    image_dir=os.path.join(chip_dir, "images"),
    label_dir=os.path.join(chip_dir, "labels"),
    chip_size=256,
    num_classes=2,  # background + building
    augment=True,
)

n_train = int(len(dataset) * 0.8)
n_val = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} chips, Val: {len(val_ds)} chips")

## 3. Train a Segmentation Model

In [ ]:
from geoai import train_segmentation

model, metrics = train_segmentation(
    train_loader=train_loader,
    val_loader=val_loader,
    architecture="Unet",           # or FPN, DeepLabV3, DeepLabV3Plus, ...
    encoder_name="resnet18",       # any timm backbone (500+ choices)
    encoder_weights="imagenet",    # start from ImageNet pretraining
    num_classes=2,
    max_epochs=20,
    learning_rate=1e-4,
    output_dir="segmentation_output",
    accelerator="auto",            # uses GPU if available, CPU otherwise
)

print("\nTraining complete.")
print(f"Best val IoU: {metrics['best_val_iou']:.4f}")

## 4. Plot Training Curves

In [ ]:
history = metrics.get("history", {})

if history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if "train_loss" in history:
        axes[0].plot(history["train_loss"], label="train")
        axes[0].plot(history["val_loss"], label="val")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training Loss")
        axes[0].legend()
    if "val_iou" in history:
        axes[1].plot(history["val_iou"])
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("IoU")
        axes[1].set_title("Validation IoU")
    plt.tight_layout()
    plt.show()
else:
    print("Check segmentation_output/logs/ for TensorBoard logs.")

## 5. Run Inference on the Full Scene

In [ ]:
from geoai import predict_segmentation
import rasterio
from rasterio.plot import show

# Tiled inference: the model processes the large image in overlapping tiles
predict_segmentation(
    model=model,
    image="sample_data/image.tif",
    output="prediction.tif",
    chip_size=256,
    overlap=64,
    batch_size=4,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
with rasterio.open("sample_data/image.tif") as src:
    show(src, ax=axes[0], title="Input image")
with rasterio.open("prediction.tif") as src:
    axes[1].imshow(src.read(1), cmap="tab10")
    axes[1].set_title("Predicted segmentation")
    axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6. Available Architectures

GeoAI exposes all segmentation_models.pytorch architectures:

| Architecture | Strengths |
|---|---|
| `Unet` | Strong skip connections, good for small object boundaries |
| `FPN` | Multi-scale features, good for objects at varying sizes |
| `DeepLabV3Plus` | State-of-the-art semantic segmentation with ASPP |
| `MAnet` | Multi-scale attention, good for complex textures |
| `PAN` | Pyramid attention, efficient for large inputs |

Combined with 500+ `timm` encoders — see https://smp.readthedocs.io/en/latest/encoders.html